# 실전 09-3: Hybrid Search (키워드 + 의미 검색 결합)

## 1. 실험 목적
- **문제점**: Vector DB는 "애플"과 "사과"가 비슷한 의미라면 같이 찾아줍니다(의미론적 검색의 장점). 하지만 제품 고유번호(예: "XQ-9908")나 특정 고유명사로 핀포인트 검색을 할 때는 오히려 못 찾는 치명적인 단점이 있습니다.
- **해결책**: 전통적인 키워드 일치 방식의 검색(BM25)과 의미론적 검색(Vector DB)을 합치는 **하이브리드 서치(Hybrid Search)**를 구현합니다. 실무 상용 서비스는 99% 이 방식을 씁니다.

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 하이브리드 서치를 위한 핵심 모듈
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

C:\Users\USER\AppData\Local\Temp\ipykernel_28288\862864824.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

## 2. 문서 로드 및 2개의 검색기(Retriever) 독립 생성
이전 노트북에서 썼던 Attention 논문 PDF를 그대로 사용합니다.

In [2]:
# PDF 파싱
pdf_path = "../data/attention_is_all_you_need.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 적당한 크기로 쪼개기
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

# -----------------------------------------------------
# 검색기 A: Vector DB (의미 기반 검색기)
# -----------------------------------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# -----------------------------------------------------
# 검색기 B: BM25 (키워드 기반 검색기)
# -----------------------------------------------------
# Elasticsearch 같은 전통적 검색 엔진의 내부 알고리즘입니다.
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3

## 3. EnsembleRetriever (하이브리드 서치 결합)
가중치(weights)를 주어 두 검색기의 결과를 섞습니다. (예: 벡터 50%, 키워드 50%)

In [3]:
# EnsembleRetriever는 내부적으로 'Reciprocal Rank Fusion(RRF)' 이라는 알고리즘을 써서 
# 두 검색기가 가져온 결과의 랭킹을 합산해 가장 신뢰할 만한 문서를 최상위로 올립니다.

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5] # 벡터 검색 50%, 키워드 검색 50% 반영
)

## 4. 실험: 극단적인 키워드로 검색해보기
논문의 저자 이름 중 한 명인 "Illia Polosukhin" 처럼 **의미(Vector)로는 찾기 힘들고 키워드(BM25)에 강력하게 걸리는 단어**를 검색해 봅니다.

In [4]:
query = "Illia Polosukhin"

print("=== [Vector DB 검색 결과] ===")
# 벡터 검색은 일리야 폴로수힌이라는 희귀한 이름의 '의미'를 찾으려다가 
# 전혀 엉뚱한 러시아어 느낌의 단어나 다른 저자 이름을 가져올 수 있습니다.
vector_result = vector_retriever.invoke(query)
for i, doc in enumerate(vector_result):
    print(f"[Result {i+1}] {doc.page_content[:150]}...")

print("\n=== [BM25 키워드 검색 결과] ===")
# 키워드 검색은 단순히 스펠링이 정확히 일치하는 문서를 무식하게 찾으므로 이런 검색에 엄청나게 강합니다.
bm25_result = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_result):
    print(f"[Result {i+1}] {doc.page_content[:150]}...")

print("\n=== [Hybrid 검색 결과 (최종)] ===")
# 하이브리드 검색기는 둘의 결과를 영리하게 섞어서 가장 최적의 문서를 뱉어냅니다.
hybrid_result = hybrid_retriever.invoke(query)
for i, doc in enumerate(hybrid_result):
    print(f"[Result {i+1}] {doc.page_content[:150]}...")

=== [Vector DB 검색 결과] ===
[Result 1] efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacin...
[Result 2] Information Processing Systems, (NIPS), 2016.
[17] Łukasz Kaiser and Ilya Sutskever. Neural GPUs learn algorithms. In International Conference
on Lear...
[Result 3] Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
...

=== [BM25 키워드 검색 결과] ===
[Result 1] Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illi...
[Result 2] Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
...
[Result 3] .
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,

## 5. 결과 해석 및 인사이트

1. **벡터 검색의 치명적 단점**: "제 핸드폰 모델명이 Galaxy S24 Ultra인데..." 같은 질문에서, 벡터 DB는 의미적으로 유사한 "아이폰 15 프로맥스" 문서를 찾아버릴 위험이 있습니다.
2. **키워드 검색의 구원**: BM25는 `S24`, `Ultra` 같은 스펠링 일치를 귀신같이 잡아냅니다.
3. **하이브리드 서치는 선택이 아닌 필수**: 실무 기업 환경(예: 사내 규정집, 부품 메뉴얼, 고객 CS)에서는 고유명사와 품번이 난무하기 때문에, 벡터 검색 단독으로는 절대 정확도가 나오지 않습니다. 반드시 위와 같이 EnsembleRetriever로 **Hybrid Search**를 구축해야 합니다.